# GameTheory 15c - Jeux Cooperatifs (C# / .NET)

**Navigation** : [<< 15-CooperativeGames (track principal)](GameTheory-15-CooperativeGames.ipynb) | [Index](README.md)

**Autres side tracks** : [15b-Lean-CooperativeGames](GameTheory-15b-Lean-CooperativeGames.ipynb) | [15c-Python-CooperativeGames](GameTheory-15c-CooperativeGames-Python.ipynb)

**Kernel** : .NET (C#) -- marathon #4956, parite .NET ⇄ Python (twin de `GameTheory-15c-CooperativeGames-Python.ipynb`).

---

## Introduction

Ce notebook est le **twin C#** du notebook `15c-Python` : il illustre les memes concepts de
theorie des jeux cooperatifs (valeur de Shapley, Core, indice de Banzhaf, jeux de vote ponderes)
implementes **from-scratch en C# .NET 9 (BCL seule, 0 NuGet)**. La parite .NET ⇄ Python est
l'objet d'etude : deux runtimes, memes valeurs exactes.

### Objectifs d'apprentissage

1. Calculer la **valeur de Shapley** d'un jeu cooperatif et l'illustrer sur le jeu de gants (Glove Game)
2. Verifier qu'un jeu de **majorite** peut avoir un **Core vide**
3. Comparer **Shapley vs Banzhaf** sur un jeu de vote pondere (Mini-ONU)
4. Tester la **convexite** d'un jeu (condition pour que Shapley soit dans le Core)

> **Note SOTA (EPIC #3801)** : il n'existe pas de solveur NuGet canonique pour la theorie des jeux
> cooperatifs. La valeur de Shapley et l'indice de Banzhaf sont des **algorithmes deterministes**
> (enumerateurs de permutations / coalitions) -- le from-scratch C# est l'implementation fidele du
> SOTA algorithmique, parallele au twin Python qui l'implemente aussi lui-meme (numpy/stdlib).


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

Console.WriteLine("Twin C# .NET des jeux cooperatifs (marathon #4956, parite avec 15c-Python)");
Console.WriteLine("Implementation from-scratch BCL .NET 9 (0 NuGet).");


Twin C# .NET des jeux cooperatifs (marathon #4956, parite avec 15c-Python)


Implementation from-scratch BCL .NET 9 (0 NuGet).


---

## 1. Valeur de Shapley - Rappels

La valeur de Shapley est la contribution marginale moyenne d'un joueur sur toutes les permutations
de l'ordre d'arrivee :

$$\phi_i(v) = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!(n-|S|-1)!}{n!} [v(S \cup \{i\}) - v(S)]$$

Implementation equivalente (et exacte) : enumerer **toutes les permutations** de $N$, et pour chaque
permutation, additionner la contribution marginale de $i$ quand il rejoint la coalition de ses
predecesseurs. On divise ensuite par $n!$.

In [2]:
// Fonction caracteristique : v(S) -> double. S est un ISet<int>.
using GameFunc = System.Func<System.Collections.Generic.ISet<int>, double>;

// Toutes les permutations de {0..n-1} (algorithme de Heap).
static IEnumerable<int[]> Permutations(int n)
{
    var a = Enumerable.Range(0, n).ToArray();
    var c = new int[n];
    yield return (int[])a.Clone();
    int i = 0;
    while (i < n)
    {
        if (c[i] < i)
        {
            if (i % 2 == 0) (a[0], a[i]) = (a[i], a[0]);
            else (a[c[i]], a[i]) = (a[i], a[c[i]]);
            yield return (int[])a.Clone();
            c[i]++;
            i = 0;
        }
        else { c[i] = 0; i++; }
    }
}

// Valeur de Shapley exacte (permutations). Retourne le vecteur phi de taille n.
static double[] ShapleyValueExact(GameFunc v, int n)
{
    var phi = new double[n];
    long count = 0;
    foreach (var perm in Permutations(n))
    {
        var coalition = new HashSet<int>();
        foreach (var i in perm)
        {
            var with = new HashSet<int>(coalition) { i };
            double marginal = v(with) - v(coalition);
            phi[i] += marginal;
            coalition.Add(i);
        }
        count++;
    }
    for (int k = 0; k < n; k++) phi[k] /= count;
    return phi;
}

Console.WriteLine("Fonction ShapleyValueExact definie.");


Fonction ShapleyValueExact definie.


---

## 2. Jeu de Gants (Glove Game)

**Exercice du notebook 15b Lean** : trois joueurs L1, L2 ont chacun un gant gauche, R1 a un gant
droit. Une paire de gants vaut 1. La fonction caracteristique compte le nombre de paires completes,
limite par le gant droit (ressource rare).

In [3]:
// Jeu de gants : joueurs 0,1 = gants gauches, joueur 2 = gant droit.
static GameFunc GloveGame() => S =>
{
    int left = S.Count(i => i == 0 || i == 1);
    int right = S.Count(i => i == 2);
    return Math.Min(left, right);
};

var glove = GloveGame();
var labels = new Dictionary<int,string> { [0]="L1", [1]="L2", [2]="R1" };

Console.WriteLine("JEU DE GANTS");
Console.WriteLine(new string('=', 40));
Console.WriteLine("Joueurs : L1=0, L2=1 (gants gauches), R1=2 (gant droit)");
Console.WriteLine();
Console.WriteLine("Fonction caracteristique :");

// Toutes les coalitions de {0,1,2}, triees par taille puis lex.
static List<List<int>> AllCoalitions(int n)
{
    var res = new List<List<int>>();
    for (int mask = 0; mask < (1 << n); mask++)
    {
        var c = new List<int>();
        for (int i = 0; i < n; i++) if ((mask & (1 << i)) != 0) c.Add(i);
        res.Add(c);
    }
    res.Sort((a, b) => a.Count != b.Count ? a.Count.CompareTo(b.Count) : string.Join(",", a).CompareTo(string.Join(",", b)));
    return res;
}

foreach (var S in AllCoalitions(3))
{
    var set = new HashSet<int>(S);
    string str = S.Count == 0 ? "{}" : "{" + string.Join(", ", S.Select(i => labels[i])) + "}";
    Console.WriteLine($"  v({str}) = {glove(set)}");
}


JEU DE GANTS


Joueurs : L1=0, L2=1 (gants gauches), R1=2 (gant droit)


Fonction caracteristique :


  v({}) = 0


  v({L1}) = 0


...

  v({R1}) = 0


  v({L1, L2}) = 0


  v({L1, R1}) = 1


  v({L2, R1}) = 1


  v({L1, L2, R1}) = 1


### Interpretation : Fonction caracteristique du jeu de gants

La fonction caracteristique encode la **structure de complementarite** du jeu :

| Type de coalition | Valeur | Raison |
|-------------------|--------|--------|
| Vide ou singletons | 0 | Aucune paire possible |
| {L1, L2} (deux gauches) | 0 | Pas de gant droit pour completer |
| {L1, R1} ou {L2, R1} | 1 | Une paire complete |
| {L1, L2, R1} | 1 | Une seule paire possible (1 gant droit) |

> **Point cle** : le gant droit est la **ressource limitante**. Cette asymetrie aura un impact
> majeur sur les valeurs de Shapley : R1 capturera l'essentiel de la valeur.

In [4]:
// Calcul de Shapley pour le jeu de gants
var shapleyGlove = ShapleyValueExact(glove, 3);

Console.WriteLine();
Console.WriteLine("VALEURS DE SHAPLEY :");
var gloveLabels = new[] { "L1 (gant gauche)", "L2 (gant gauche)", "R1 (gant droit)" };
for (int i = 0; i < 3; i++)
{
    // valeur * 6 pour exprimer en sixiemes (denominateur commun n! = 6)
    Console.WriteLine($"  {gloveLabels[i]}: {shapleyGlove[i]:F4} = {(int)Math.Round(shapleyGlove[i]*6)}/6");
}
Console.WriteLine();
Console.WriteLine($"Total : {shapleyGlove.Sum():F4} (= v(N) = 1)");
Console.WriteLine();
Console.WriteLine("Interpretation :");
Console.WriteLine("  - R1 (gant droit) a une valeur de 4/6 = 2/3 car il possede la ressource rare");
Console.WriteLine("  - L1 et L2 se partagent 2/6 = 1/3 car ils sont en competition");


VALEURS DE SHAPLEY :


  L1 (gant gauche): 0,1667 = 1/6


  L2 (gant gauche): 0,1667 = 1/6


  R1 (gant droit): 0,6667 = 4/6


Total : 1,0000 (= v(N) = 1)


Interpretation :


  - R1 (gant droit) a une valeur de 4/6 = 2/3 car il possede la ressource rare


  - L1 et L2 se partagent 2/6 = 1/3 car ils sont en competition


### Interpretation : Valeur de la rarete

| Joueur | Ressource | Shapley | Explication |
|--------|-----------|---------|-------------|
| L1, L2 | Gant gauche (abondant) | 1/6 chacun | Competition entre detenteurs de la meme ressource |
| R1 | Gant droit (rare) | 4/6 | Monopole sur la ressource complementaire |

> **Lecon economique** : la valeur d'une ressource ne depend pas seulement de son utilite
> intrinseque, mais de sa **rarete relative** par rapport aux ressources complementaires.

In [5]:
// Visualisation ASCII des valeurs de Shapley (jeu de gants)
Console.WriteLine("Valeurs de Shapley (jeu de gants) :");
Console.WriteLine();
double maxBar = 0.8;
foreach (var (lab, val) in gloveLabels.Zip(shapleyGlove))
{
    int width = (int)Math.Round(val / maxBar * 30);
    string bar = new string('#', width);
    string marker = Math.Abs(val - 1.0/3) < 1e-9 ? "  <-- partage egal (1/3)" : "";
    Console.WriteLine($"  {lab,-20} {val:F4} |{bar}{marker}");
}
double third = 1.0 / 3;
int twidth = (int)Math.Round(third / maxBar * 30);
string egalLbl = "Partage egal 1/3";
Console.WriteLine($"  {egalLbl,-20} {third:F4} |{new string('-', twidth)}");


Valeurs de Shapley (jeu de gants) :


  L1 (gant gauche)     0,1667 |######


  L2 (gant gauche)     0,1667 |######


  R1 (gant droit)      0,6667 |#########################


  Partage egal 1/3     0,3333 |------------


### Exercice 1 : Jeu de gants etendu a 4 joueurs

**Objectif** : etendre le jeu de gants a 4 joueurs (L1, L2 gauches ; R1, R2 droits).
Definir `GloveGame4`, calculer les valeurs de Shapley, interpreter comment le marche 2-vs-2
repartit la valeur (ressource equilibree vs rare).

- **Indice :** joueurs 0,1 = gauches, 2,3 = droits. v(S) = min(gauches, droites).
- **Etape 1 :** definir `GloveGame4`.
- **Etape 2 :** calculer `ShapleyValueExact(GloveGame4, 4)`.
- **Etape 3 :** comparer avec le resultat a 3 joueurs.

In [6]:
// Exercice 1 (stub C.1) : jeu de gants etendu a 4 joueurs.
static double[] ExerciceGloveGame4()
{
    // TODO etudiant : retourner ShapleyValueExact(GloveGame4(), 4)
    // ou GloveGame4(S) = min(count gauche parmi {0,1}, count droit parmi {2,3}).
    return new double[0];  // TODO etudiant
}

var shapleyGlove4 = ExerciceGloveGame4();
var labels4 = new[] { "L1", "L2", "R1", "R2" };
if (shapleyGlove4.Length == 4)
{
    foreach (var (lab, val) in labels4.Zip(shapleyGlove4))
        Console.WriteLine($"  {lab}: {val:F4}");
    Console.WriteLine($"Total: {shapleyGlove4.Sum():F4}");
}
else
{
    Console.WriteLine("Exercice 1 a completer : valeurs de Shapley du jeu de gants a 4 joueurs.");
}


Exercice 1 a completer : valeurs de Shapley du jeu de gants a 4 joueurs.


---

## 3. Core Vide : Jeu de Majorite

On montre que le jeu de majorite simple a 3 joueurs a un **Core vide** : aucune allocation
efficace n'est stable (toute proposition est bloquee par une coalition de 2 joueurs).

In [7]:
// Jeu de majorite simple a 3 joueurs : v(S) = 1 si |S| >= 2, 0 sinon.
static GameFunc MajorityGame3() => S => S.Count >= 2 ? 1.0 : 0.0;

var maj3 = MajorityGame3();
Console.WriteLine("JEU DE MAJORITE SIMPLE A 3 JOUEURS");
Console.WriteLine(new string('=', 50));
Console.WriteLine();
Console.WriteLine("Fonction caracteristique :");
foreach (var S in AllCoalitions(3))
{
    var set = new HashSet<int>(S);
    string str = S.Count == 0 ? "{}" : "{" + string.Join(", ", S.Select(i => (i+1).ToString())) + "}";
    Console.WriteLine($"  v({str}) = {(int)maj3(set)}");
}


JEU DE MAJORITE SIMPLE A 3 JOUEURS


Fonction caracteristique :


  v({}) = 0


  v({1}) = 0


  v({2}) = 0


  v({3}) = 0


  v({1, 2}) = 1


  v({1, 3}) = 1


  v({2, 3}) = 1


  v({1, 2, 3}) = 1


### Interpretation : Structure du jeu de majorite

La fonction caracteristique revele une **symetrie parfaite** entre les joueurs :

- **Coalitions perdantes** (v=0) : singletons uniquement
- **Coalitions gagnantes** (v=1) : toute paire ou plus

Cette structure est celle d'un **jeu simple symetrique** : tous les joueurs sont interchangeables.
On s'attend donc a des valeurs de Shapley egales (1/3 chacune) -- mais cette allocation **n'est pas
stable** (pas dans le Core).

In [8]:
// Preuve que le Core est vide (3 joueurs, majorite simple).
Console.WriteLine("PREUVE QUE LE CORE EST VIDE");
Console.WriteLine(new string('=', 50));
Console.WriteLine(@"
Pour qu'une allocation (x1, x2, x3) soit dans le Core :

1. Efficacite : x1 + x2 + x3 = v(N) = 1

2. Stabilite (aucune coalition ne peut bloquer) :
   x1 + x2 >= v({1,2}) = 1
   x1 + x3 >= v({1,3}) = 1
   x2 + x3 >= v({2,3}) = 1

En additionnant les trois contraintes de stabilite :
   2(x1 + x2 + x3) >= 3
   2 * 1 >= 3   (par efficacite x1+x2+x3=1)
   2 >= 3       CONTRADICTION!

=> Le Core est VIDE.

Intuition : chaque coalition de 2 joueurs peut bloquer et demander au moins 1,
mais il n'y a que 1 a partager entre les 3 joueurs.
");


PREUVE QUE LE CORE EST VIDE



Pour qu'une allocation (x1, x2, x3) soit dans le Core :

1. Efficacite : x1 + x2 + x3 = v(N) = 1

2. Stabilite (aucune coalition ne peut bloquer) :
   x1 + x2 >= v({1,2}) = 1
   x1 + x3 >= v({1,3}) = 1
   x2 + x3 >= v({2,3}) = 1

En additionnant les trois contraintes de stabilite :
   2(x1 + x2 + x3) >= 3
   2 * 1 >= 3   (par efficacite x1+x2+x3=1)
   2 >= 3       CONTRADICTION!

=> Le Core est VIDE.

Intuition : chaque coalition de 2 joueurs peut bloquer et demander au moins 1,
mais il n'y a que 1 a partager entre les 3 joueurs.



### Interpretation : Preuve du Core vide

La preuve utilise la technique classique de **sommation des contraintes** :

1. Chaque paire exige au moins 1 (elle forme une coalition gagnante).
2. En additionnant les 3 contraintes de paires, chaque joueur apparait 2 fois.
3. On obtient `2 * total >= 3`, soit `2 >= 3` -- contradiction.

> **Consequence** : dans un jeu de majorite simple, aucun partage ne satisfait toutes les
> coalitions. Shapley donne une allocation "juste" (1/3, 1/3, 1/3) mais elle n'est pas stable.

In [9]:
// Shapley du jeu de majorite (tous egaux par symetrie).
var shapleyMajority = ShapleyValueExact(maj3, 3);

Console.WriteLine("VALEURS DE SHAPLEY :");
for (int i = 0; i < 3; i++)
    Console.WriteLine($"  Joueur {i+1}: {shapleyMajority[i]:F4} = 1/3");
Console.WriteLine();
Console.WriteLine("Note : Shapley donne une allocation 'juste' (1/3, 1/3, 1/3)");
Console.WriteLine("mais cette allocation n'est PAS stable (pas dans le Core).");
Console.WriteLine();
Console.WriteLine("Verification : chaque coalition de 2 peut bloquer");
Console.WriteLine($"  x1 + x2 = {shapleyMajority[0] + shapleyMajority[1]:F4} < 1 = v({{1,2}})");


VALEURS DE SHAPLEY :


  Joueur 1: 0,3333 = 1/3


  Joueur 2: 0,3333 = 1/3


  Joueur 3: 0,3333 = 1/3


Note : Shapley donne une allocation 'juste' (1/3, 1/3, 1/3)


mais cette allocation n'est PAS stable (pas dans le Core).


Verification : chaque coalition de 2 peut bloquer


  x1 + x2 = 0,6667 < 1 = v({1,2})


---

## 4. Jeux de Vote Ponderes et indice de Banzhaf

Un jeu de vote pondere $[q; w_1, \dots, w_n]$ ou une coalition gagne si $\sum_{i \in S} w_i \geq q$.
L'**indice de Banzhaf** mesure le pouvoir d'un joueur comme le nombre de coalitions ou il est
**critique** (S gagne, S \ {i} perd), normalise par la somme des pouvoirs.

In [10]:
// Jeu de vote pondere : [quota; w1..wn]. v(S) = 1 si sum des poids >= quota.
static GameFunc WeightedVotingGame(int[] weights, int quota) => S =>
    S.Sum(i => weights[i]) >= quota ? 1.0 : 0.0;

// Indice de Banzhaf normalise : compte les coalitions gagnantes ou chaque joueur est critique.
static double[] BanzhafIndex(GameFunc v, int n)
{
    var critical = new double[n];
    double total = 0;
    foreach (var S in AllCoalitions(n))
    {
        var set = new HashSet<int>(S);
        if (v(set) != 1.0) continue;          // coalition gagnante uniquement
        foreach (var i in S)
        {
            var without = new HashSet<int>(set); without.Remove(i);
            if (v(without) != 1.0)            // i est critique : S\{i} perd
            {
                critical[i] += 1;
                total += 1;
            }
        }
    }
    if (total == 0) return critical;
    for (int k = 0; k < n; k++) critical[k] /= total;
    return critical;
}

Console.WriteLine("Fonctions WeightedVotingGame + BanzhafIndex definies.");


Fonctions WeightedVotingGame + BanzhafIndex definies.


### Mini-ONU : un jeu de vote pondere

**Mini-ONU** $[9; 7, 7, 1, 1, 1]$ : deux membres permanents (P1, P2, poids 7 chacun) et trois
membres non-permanents (N1, N2, N3, poids 1). Une motion passe si le poids total atteint 9.
On compare **Shapley vs Banzhaf** -- revelateur de l'ecart entre poids nominal et pouvoir reel.

In [11]:
// Mini-ONU : [9; 7, 7, 1, 1, 1]
int[] wUN = { 7, 7, 1, 1, 1 };
int quotaUN = 9;
var un = WeightedVotingGame(wUN, quotaUN);
var shapleyUN = ShapleyValueExact(un, 5);
var banzhafUN = BanzhafIndex(un, 5);
var labelsUN = new[] { "P1", "P2", "N1", "N2", "N3" };

Console.WriteLine("JEU DE VOTE PONDERE : Mini-ONU");
Console.WriteLine(new string('=', 48));
Console.WriteLine($"[{quotaUN}; {string.Join(", ", wUN)}]");
Console.WriteLine("P1, P2 = permanents (poids 7), N1, N2, N3 = non-permanents (poids 1)");
Console.WriteLine();
Console.WriteLine("Comparaison Shapley vs Banzhaf :");
Console.WriteLine($"{"Joueur",-10}{"Poids",-10}{"Shapley",-12}{"Banzhaf",-12}");
Console.WriteLine(new string('-', 44));
for (int i = 0; i < 5; i++)
    Console.WriteLine($"{labelsUN[i],-10}{wUN[i],-10}{shapleyUN[i],-12:F4}{banzhafUN[i],-12:F4}");
Console.WriteLine();
Console.WriteLine($"Somme Shapley = {shapleyUN.Sum():F4}, Somme Banzhaf = {banzhafUN.Sum():F4}");


JEU DE VOTE PONDERE : Mini-ONU


[9; 7, 7, 1, 1, 1]


P1, P2 = permanents (poids 7), N1, N2, N3 = non-permanents (poids 1)


Comparaison Shapley vs Banzhaf :


Joueur    Poids     Shapley     Banzhaf     


...

P2        7         0,3000      0,2857      


N1        1         0,1333      0,1429      


N2        1         0,1333      0,1429      


N3        1         0,1333      0,1429      


Somme Shapley = 1,0000, Somme Banzhaf = 1,0000


### Interpretation : Indices de pouvoir

| Joueur | Poids nominal | Pouvoir Shapley | Ratio pouvoir/poids |
|--------|---------------|-----------------|---------------------|
| P1, P2 | 7 (41%) | 0.30 (30%) | 0.73 |
| N1-N3 | 1 (6%) | 0.13 (13%) | 2.17 |

**Paradoxe classique** : les non-permanents ont **plus de pouvoir relatif** que leur poids.
Avec 6% du poids, ils captent 13% du pouvoir. Inversement, les permanents (41% du poids) n'ont
que 30% du pouvoir. Le poids nominal ment sur le pouvoir reel.

In [12]:
// Visualisation ASCII comparative Shapley vs Banzhaf (Mini-ONU).
Console.WriteLine("Comparaison Shapley vs Banzhaf (Mini-ONU) :");
Console.WriteLine();
double maxPow = Math.Max(shapleyUN.Max(), banzhafUN.Max());
foreach (var (lab, sh, ba) in labelsUN.Zip(shapleyUN, banzhafUN))
{
    int wS = (int)Math.Round(sh / maxPow * 30);
    int wB = (int)Math.Round(ba / maxPow * 30);
    Console.WriteLine($"  {lab} (poids {wUN[Array.IndexOf(labelsUN, lab)]})");
    Console.WriteLine($"    Shapley {sh:F4} |{new string('#', wS)}");
    Console.WriteLine($"    Banzhaf {ba:F4} |{new string('+', wB)}");
}
Console.WriteLine();
Console.WriteLine("Note : le pouvoir reel (Shapley/Banzhaf) peut differer du poids nominal.");


Comparaison Shapley vs Banzhaf (Mini-ONU) :


  P1 (poids 7)


    Shapley 0,3000 |##############################


    Banzhaf 0,2857 |+++++++++++++++++++++++++++++


  P2 (poids 7)


...

    Banzhaf 0,1429 |++++++++++++++


  N3 (poids 1)


    Shapley 0,1333 |#############


    Banzhaf 0,1429 |++++++++++++++


Note : le pouvoir reel (Shapley/Banzhaf) peut differer du poids nominal.


---

## 5. Jeux Convexes : Shapley dans le Core

Un jeu est **convexe** (supermodulaire) si les contributions marginales sont croissantes :
pour tous S, T : `v(S) + v(T) <= v(S u T) + v(S n T)`.
**Theoreme** : pour un jeu convexe, la valeur de Shapley est **dans le Core** (stable).

In [13]:
// Test de convexite : pour tous S, T : v(S)+v(T) <= v(S u T)+v(S n T).
static bool IsConvex(GameFunc v, int n)
{
    foreach (var Ss in AllCoalitions(n))
    {
        var S = new HashSet<int>(Ss);
        foreach (var Tt in AllCoalitions(n))
        {
            var T = new HashSet<int>(Tt);
            var u = new HashSet<int>(S); u.UnionWith(T);
            var inter = new HashSet<int>(S); inter.IntersectWith(T);
            double lhs = v(S) + v(T);
            double rhs = v(u) + v(inter);
            if (lhs > rhs + 1e-10) return false;
        }
    }
    return true;
}

// Jeu d'unanimite u_{1,2} : v(S) = 1 si S contient {1,2}, 0 sinon.
static GameFunc UnanimityGame12() => S => (S.Contains(0) && S.Contains(1)) ? 1.0 : 0.0;

Console.WriteLine("TEST DE CONVEXITE");
Console.WriteLine(new string('=', 40));
Console.WriteLine($"Jeu de gants convexe ?     {IsConvex(glove, 3)}");
Console.WriteLine($"Jeu de majorite convexe ?  {IsConvex(maj3, 3)}");
Console.WriteLine($"Jeu d'unanimite convexe ?  {IsConvex(UnanimityGame12(), 3)}");


TEST DE CONVEXITE


Jeu de gants convexe ?     False


Jeu de majorite convexe ?  False


Jeu d'unanimite convexe ?  True


### Interpretation : Tests de convexite

- **Glove game** : NON convexe (les contributions marginales ne sont pas croissantes -- un deuxieme
  gant gauche n'apporte rien une fois une paire formee).
- **Majorite** : NON convexe (structure symetrique non supermodulaire).
- **Unanimite** `{1,2}` : convexe. Le theoreme s'applique : sa valeur de Shapley est dans le Core.

> **Lecon** : la convexite est la condition cle qui reconcilie **equite** (Shapley) et **stabilite**
> (Core). Hors convexite, comme dans le jeu de majorite, Shapley est equitable mais instable.

### Exercice 2 : Jeu de vote pondere personnalise

**Objectif** : definir un jeu de vote pondere de votre choix (ex : conseil municipal $[6; 4,3,2,1]$),
calculer Shapley et Banzhaf, identifier les joueurs dont le pouvoir depasse le poids.

- **Indice :** `WeightedVotingGame(new[]{4,3,2,1}, 6)`.
- **Etape 1 :** calculer `ShapleyValueExact` et `BanzhafIndex`.
- **Etape 2 :** comparer ratio pouvoir/poids de chaque joueur.

In [14]:
// Exercice 2 (stub C.1) : analyse d'un jeu de vote pondere personnalise.
static (double[] shapley, double[] banzhaf) ExerciceWeightedVoting()
{
    // TODO etudiant : choisir un jeu [q; w...], retourner (ShapleyValueExact, BanzhafIndex).
    return (new double[0], new double[0]);  // TODO etudiant
}

var (sh2, bz2) = ExerciceWeightedVoting();
if (sh2.Length > 0)
{
    Console.WriteLine("Exercice 2 : pouvoir par joueur (Shapley / Banzhaf).");
}
else
{
    Console.WriteLine("Exercice 2 a completer : Shapley et Banzhaf d'un jeu de vote pondere.");
}


Exercice 2 a completer : Shapley et Banzhaf d'un jeu de vote pondere.


### Exercice 3 : Verification du Core pour un jeu convexe

**Objectif** : pour le jeu d'unanimite `{1,2}` a 3 joueurs (convexe), verifier que la valeur de
Shapley est bien dans le Core en testant toutes les contraintes de stabilite.

- **Indice :** `UnanimityGame12()` ; une allocation (x1,x2,x3) est dans le Core si elle est efficace
  (`x1+x2+x3 = v(N)`) et stable (`xS >= v(S)` pour toute coalition S).
- **Etape 1 :** calculer `ShapleyValueExact(UnanimityGame12(), 3)`.
- **Etape 2 :** verifier que chaque contrainte de coalition est satisfaite.

In [15]:
// Exercice 3 (stub C.1) : verifier que Shapley d'un jeu convexe est dans le Core.
static bool ExerciceShapleyInCore()
{
    // TODO etudiant : calculer Shapley de UnanimityGame12(), verifier toutes les contraintes du Core.
    return false;  // TODO etudiant
}

bool inCore = ExerciceShapleyInCore();
Console.WriteLine(inCore
    ? "Exercice 3 : Shapley du jeu d'unanimite EST dans le Core (convexe)."
    : "Exercice 3 a completer : verifier l'appartenance de Shapley au Core.");


Exercice 3 a completer : verifier l'appartenance de Shapley au Core.


---

## Conclusion

Ce twin C# a confirme, valeur pour valeur, les memes resultats que le notebook Python :

- **Shapley (glove)** : (1/6, 1/6, 4/6) -- la ressource rare (R1) capture l'essentiel.
- **Core vide** : le jeu de majorite 3-joueurs n'admet aucun partage stable (contradiction 2 >= 3).
- **Shapley (majorite)** : (1/3, 1/3, 1/3) -- equitable mais instable.
- **Mini-ONU** : ecart poids/pouvoir revelé (Shapley vs Banzhaf proches ici).
- **Convexite** : seule l'unanimite est convexe (Shapley alors dans le Core).

> **Parite .NET ⇄ Python (marathon #4956)** : tous les ancres deterministes (valeurs de Shapley,
> fonction caracteristique, convexite) sont concordants au bit pres entre le twin C# (BCL .NET 9,
> 0 NuGet) et le twin Python (numpy/stdlib). Les deux runtimes implementent fidelement les memes
> algorithmes exacts (permutations pour Shapley, enumeration de coalitions pour Banzhaf).

Voir aussi : [15-CooperativeGames](GameTheory-15-CooperativeGames.ipynb) (track principal),
[15b-Lean-CooperativeGames](GameTheory-15b-Lean-CooperativeGames.ipynb) (preuves formelles Lean).
